# A contra(luz): análisis y veeduría de la contratación pública en Colombia

**Curso:** Taller de Programación Aplicada a Python (*MECA 4110 / EECO 5617*) — Universidad de los Andes, 2026-II  
**Integrantes — Grupo 3:**
- **Diego Andrés Niño Estupiñán** — Código: `201719748`
- **Juan Sebastián Puerto Sanz** — Código: `201532975`
- **Nathalia Alexandra Acero Pérez** — Código: `202524470`

---

### Pregunta central de investigación
> *¿Qué tanto se desvía la contratación ejecutada en SECOP II frente a lo planeado en el Plan Anual de Adquisiciones (PAA) entre 2022 y 2025 en seis entidades nacionales del nivel central (ANCP, MinHacienda, MinAmbiente, MinEducación, MinCulturas y DNP), y en qué entidades, modalidades de contratación, proveedores o segmentos UNSPSC se concentra dicha desviación?*

### Trazabilidad frente a las recomendaciones del equipo pedagógico (`R1–R8`)

| Código | Recomendación incorporada | Sección y evidencia en este tutorial |
|---|---|---|
| **R1** | Pregunta enfocada en el fenómeno económico (no en construir una herramienta) | Encabezado y Sección 1 |
| **R2** | Cruce efectivo entre el PAA y los contratos electrónicos de SECOP II | Sección 3 (`src/limpieza_cruce.py`) |
| **R3** | Extracción vía API Socrata SODA (`datos.gov.co`) con caché reproducible offline | Sección 2 (`src/ingesta.py`) |
| **R4** | Muestra concreta de 6 entidades nacionales del nivel central en `2022–2025` | Sección 1 (`27.160` ítems PAA y `25.317` contratos) |
| **R5** | Indicadores explícitos desde el inicio (`% ejecutado`, directa, `HHI`, brechas UNSPSC) | Sección 4 (`src/indicadores.py`) |
| **R6** | Método exacto de cruce UNSPSC justificado empíricamente (`Clase` 6d y `Segmento` 2d) | Sección 3 (`99,99%` PAA y `99,80%` Contratos) |
| **R7** | Especificación explícita de qué calcula cada etapa de la secuencia | Introducciones de las Secciones 1 a 6 |
| **R8** | Análisis económico sustantivo con tablero interactivo en Streamlit como síntesis | Secciones 4, 5 y 6 (`app_streamlit.py`) |

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import (
    ANIOS_ESTUDIO,
    DATASET_CONTRATOS,
    DATASET_PAA_DETALLE,
    DATASET_PAA_ENCABEZADO,
    ENTIDADES_COMPLETO,
    FACTOR_DEFLACTOR_IPC_2025,
)
from src.indicadores import ejecutar_calculo_indicadores
from src.ingesta import cargar_datos_crudos
from src.limpieza_cruce import ejecutar_limpieza_y_cruce
from src.visualizacion import (
    graficar_concentracion_hhi,
    graficar_contratacion_directa_entidades,
    graficar_distribucion_franjas_cuantia,
    graficar_evolucion_tasa_ejecucion,
    graficar_planeado_vs_ejecutado_entidad,
    graficar_rubro5_antes_despues_ipc,
    graficar_rubro5_ejecucion_vs_duncan,
    graficar_top_brechas_unspsc,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.options.display.float_format = "{:,.2f}".format

RUTA_DATOS_RAW = Path("data/raw")
assert RUTA_DATOS_RAW.exists(), "Ejecute este notebook desde la carpeta raíz del proyecto."
print(f"Entorno listo | RANDOM_SEED = {RANDOM_SEED} | Ruta relativa verificada: {RUTA_DATOS_RAW}")

Entorno listo | RANDOM_SEED = 42 | Ruta relativa verificada: data\raw


## 1. Marco Institucional, Muestra de Entidades y Fuentes Oficiales
Definida la pregunta central, esta sección delimita las seis entidades nacionales estudiadas entre 2022 y 2025 y presenta los cuatro activos oficiales que permiten contrastar planeación anual contra ejecución.

En Colombia, el **Plan Anual de Adquisiciones (PAA)** es un instrumento de planeación indicativo y modificable (Decreto 1082 de 2015, arts. 2.2.1.1.1.4.1 a 4.4), mientras que los **Contratos Electrónicos de SECOP II** registran compromisos jurídicos efectivos regidos por el Estatuto General de Contratación (Ley 80 de 1993 y Ley 1150 de 2007). Para evitar distorsiones por redes territoriales desconcentradas o planes en borrador, seleccionamos seis entidades del nivel central con estado `Publicado` en las cuatro vigencias.

In [2]:
tabla_entidades = pd.DataFrame(
    [
        {
            "codigo_entidad": cod,
            "sigla": info["sigla"],
            "entidad": info["nombre_corto"],
            "nit": info["nit"],
            "sector_analisis": info["sector_analisis"],
        }
        for cod, info in ENTIDADES_COMPLETO.items()
    ]
)
tabla_fuentes = pd.DataFrame(
    [
        {"activo": nombre, "identificador_socrata": codigo, "vigencias": f"{min(ANIOS_ESTUDIO)}-{max(ANIOS_ESTUDIO)}"}
        for nombre, codigo in [
            ("SECOP II - PAA Encabezado", DATASET_PAA_ENCABEZADO),
            ("SECOP II - PAA Detalle", DATASET_PAA_DETALLE),
            ("SECOP II - Contratos Electrónicos", DATASET_CONTRATOS),
        ]
    ]
    + [
        {
            "activo": "Clasificador UNSPSC v14.1",
            "identificador_socrata": "Colombia Compra Eficiente (UNSPSC v14.1)",
            "vigencias": "Catálogo jerárquico oficial",
        }
    ]
)
display(tabla_entidades)
display(tabla_fuentes)

,codigo_entidad,sigla,entidad,nit,sector_analisis
0,700037070,ANCP,ANCP (Colombia Compra Eficiente),900514813,Rectoría de Compras Públicas
1,700065089,MinHacienda,Ministerio de Hacienda y Crédito Público,899999090,Hacienda y Política Fiscal
2,700209018,MinAmbiente,Ministerio de Ambiente y Desarrollo Sostenible,830115395,Ambiente y Desarrollo Sostenible
3,700066053,MinEducación,Ministerio de Educación Nacional,899999001,Educación
4,700637010,MinCulturas,"Ministerio de las Culturas, las Artes y los Sa...",830034348,Cultura
5,700220049,DNP,Departamento Nacional de Planeación,899999011,Planeación Nacional


,activo,identificador_socrata,vigencias
0,SECOP II - PAA Encabezado,b6m4-qgqv,2022-2025
1,SECOP II - PAA Detalle,9sue-ezhx,2022-2025
2,SECOP II - Contratos Electrónicos,jbjy-vk9h,2022-2025
3,Clasificador UNSPSC v14.1,Colombia Compra Eficiente (UNSPSC v14.1),Catálogo jerárquico oficial


## 2. Adquisición Reproducible vía API Socrata y Verificación de Grano
Establecidas las fuentes oficiales, cargamos los registros extraídos programáticamente vía API Socrata SODA (con respaldo offline en `data/raw/`) y verificamos sus dimensiones, tipos y cuadre presupuestal.

In [3]:
# Para refrescar desde datos.gov.co en línea: python -m src.ingesta --modo completo --actualizar-api
# Por reproducibilidad autónoma (Rubro 1 y Rubro 2), leemos la extracción congelada en data/raw/:
datos_crudos = cargar_datos_crudos(usar_cache_local=True)

resumen_crudo = pd.DataFrame(
    [
        {
            "tabla_cruda": nombre,
            "filas": len(df),
            "columnas": df.shape[1],
            "columnas_tipo_texto": int(df.select_dtypes(include=["object", "string"]).shape[1]),
            "filas_duplicadas_exactas": int(df.duplicated().sum()),
            "celdas_faltantes_pct": round(100.0 * float(df.isna().sum().sum()) / max(df.size, 1), 2),
        }
        for nombre, df in datos_crudos.items()
        if isinstance(df, pd.DataFrame)
    ]
)
display(resumen_crudo)

,tabla_cruda,filas,columnas,columnas_tipo_texto,filas_duplicadas_exactas,celdas_faltantes_pct
0,paa_encabezado,24,13,8,0,0.00
1,paa_detalle,27160,23,18,0,0.00
2,contratos,25317,31,26,0,0.96
3,unspsc,49022,8,8,0,0.00


La inspección confirma tres propiedades estructurales de las fuentes abiertas de Colombia Compra Eficiente:
1. **`paa_encabezado` (`24` filas = `6 entidades × 4 años`)** conserva exactamente la última versión anual publicada de cada entidad (entre `17` y `82` modificaciones por año).
2. **`paa_detalle` (`27.160` filas)** desagrega los ítems de esa última versión publicada y distingue entre `valor_total_esperado` (incluye vigencias futuras plurianuales, Ley 819 de 2003) y `valor_esperado_de_presupuesto` (apropiación de la vigencia corriente).
3. **`contratos_secop2` (`25.317` filas)** y **`unspsc` (`49.022` productos)** vienen tipificados originalmente como texto desde la API JSON/CSV, por lo que requieren conversión numérica explícita y auditoría de valores extremos en la siguiente sección.

## 3. Auditoría de Calidad, Limpieza Justificada y Cruce Jerárquico UNSPSC
Verificado el grano de las tablas crudas, auditamos faltantes, duplicados y cuantías atípicas, demostramos por qué el cruce UNSPSC debe hacerse a nivel de Clase (6 dígitos) e inspeccionamos las franjas de valor.

In [4]:
_ = ejecutar_limpieza_y_cruce(usar_cache_local=True)
auditoria = json.loads(Path("data/processed/auditoria_calidad.json").read_text(encoding="utf-8"))

tabla_auditoria = pd.DataFrame(
    [
        ("PAA: duplicados exactos de id eliminados", auditoria["paa_detalle"]["duplicados_id_eliminados"]),
        ("PAA: modalidades 'No Definido' crudas", auditoria["paa_detalle"]["items_modalidad_cruda_no_definida"]),
        ("PAA: modalidades rescatadas desde causal legal (Ley 1150/2007)", auditoria["paa_detalle"]["items_modalidad_recuperada_de_causal"]),
        ("PAA: cuasi-duplicado administrativo >= $50.000M (MinAmbiente 2023)", auditoria["paa_detalle"]["posibles_duplicados_atipicos_50mm"]),
        ("PAA: cuadre exacto sum(detalle) vs. encabezado (máx. diferencia COP)", round(auditoria["cuadre_exacto_paa_encabezado_max_diff_cop"], 6)),
        ("Contratos: registros en $0 (AMP / convenios sin erogación)", auditoria["contratos_secop2"]["contratos_valor_exacto_cero"]),
        ("Contratos: cuantías simbólicas ($0,01 a $1.000 COP)", auditoria["contratos_secop2"]["contratos_cuantia_simbolica_hasta_1000"]),
        ("Contratos: válidos con cuantía real (> $1.000 COP)", auditoria["contratos_secop2"]["contratos_validos_con_cuantia"]),
    ],
    columns=["regla_de_calidad_y_limpieza", "resultado_auditado"],
)

tabla_unspsc_r6 = pd.DataFrame(
    [
        {
            "nivel_jerarquico_unspsc": "Producto exacto (8 dígitos)",
            "match_paa_pct": auditoria["paa_detalle"]["tasa_match_unspsc_producto_8d_pct"],
            "match_contratos_pct": auditoria["contratos_secop2"]["tasa_match_unspsc_producto_8d_pct"],
            "decision_metodologica": "Descartado: entidades reportan códigos terminados en '00' (nivel Clase)",
        },
        {
            "nivel_jerarquico_unspsc": "Clase (6 dígitos) y Segmento (2 dígitos)",
            "match_paa_pct": auditoria["paa_detalle"]["tasa_match_unspsc_clase_segmento_pct"],
            "match_contratos_pct": auditoria["contratos_secop2"]["tasa_match_unspsc_clase_segmento_pct"],
            "decision_metodologica": "Adoptado: cobertura cuasi-universal sin pérdida de información",
        },
    ]
)

display(tabla_auditoria)
display(tabla_unspsc_r6)

,regla_de_calidad_y_limpieza,resultado_auditado
0,PAA: duplicados exactos de id eliminados,0.00
1,PAA: modalidades 'No Definido' crudas,"2,733.00"
2,PAA: modalidades rescatadas desde causal legal...,"2,251.00"
3,PAA: cuasi-duplicado administrativo >= $50.000...,1.00
4,PAA: cuadre exacto sum(detalle) vs. encabezado...,0.00
5,Contratos: registros en $0 (AMP / convenios si...,951.00
6,"Contratos: cuantías simbólicas ($0,01 a $1.000...",13.00
7,Contratos: válidos con cuantía real (> $1.000 ...,"24,353.00"


,nivel_jerarquico_unspsc,match_paa_pct,match_contratos_pct,decision_metodologica
0,Producto exacto (8 dígitos),39.55,36.37,Descartado: entidades reportan códigos termina...
1,Clase (6 dígitos) y Segmento (2 dígitos),99.99,99.80,Adoptado: cobertura cuasi-universal sin pérdid...


In [5]:
tablas_ind = ejecutar_calculo_indicadores()
fig_franjas = graficar_distribucion_franjas_cuantia(tablas_ind["franjas_cuantia"], sigla_entidad="TODAS")
fig_franjas

### Hallazgos sustantivos de la auditoría de datos y cuantías
- **Justificación empírica del nivel de cruce UNSPSC (`R6`):** cruzar a 8 dígitos perdería más del `60%` de ambas bases (`39,55%` y `36,37%`). Al extraer la cadena de 8 dígitos tras el prefijo `V1.` y cruzar por **`codigo_clase` (primeros 6 dígitos)** agregando por **`codigo_segmento` (primeros 2 dígitos, 56 sectores)**, el emparejamiento sube a **`99,99%` en PAA** y **`99,80%` en Contratos** (los `51` contratos `"UNSPECIFIED"` quedan rotulados como `"Sin clasificar"`).
- **Rescate de modalidades y control de atípicos:** recuperar la modalidad desde `causal_de_contratacion` rescata **`2.251` ítems del PAA (`$1.975,68 MM`)** que figuraban como `"No Definido"`. Asimismo, detectar la reedición administrativa de `$333,54 MM` en **MinAmbiente 2023** (*Fondo para la Vida y la Biodiversidad*) corrige un sesgo de `-18,0 pp` en su tasa de ejecución anual (`32,9%` bruta vs. `50,9%` ajustada).
- **Asimetría extrema y cuantías en `$0` o simbólicas:** no existen contratos negativos (`< $0`); los `951` contratos en `$0` corresponden mayoritariamente a **Acuerdos Marco de Precios de la ANCP (`565`)** y convenios marco cuya erogación ocurre después en órdenes de compra, mientras que `13` registros entre `$0,01` y `$1.000 COP` son valores simbólicos de parametrización en SECOP II. Entre los `24.353` contratos con cuantía real, el `67,9%` son contratos estándar (`$10M–$100M`), pero apenas el **`2,8%` de contratos de gran cuantía (`> $1.000M`) concentra el `75,4%` de todo el dinero ejecutado**.

## 4. Indicadores Económicos de Ejecución, Modalidades, Proveedores y UNSPSC
Con las bases depuradas y homologadas al 99,8%+, calculamos e interpretamos los cuatro indicadores centrales para identificar en qué entidades, modalidades, contratistas y segmentos UNSPSC se concentra la desviación.

### 4.1 Magnitud global y trayectoria anual de ejecución frente al PAA

In [6]:
df_resumen = tablas_ind["resumen_anual"]
fig_paa_ejec = graficar_planeado_vs_ejecutado_entidad(df_resumen, usar_base_ajustada=True)
fig_paa_ejec

In [7]:
fig_evolucion = graficar_evolucion_tasa_ejecucion(df_resumen, usar_base_ajustada=True)
fig_evolucion

**Interpretación (Indicador 1 — Tasa de ejecución):** En el agregado `2022–2025`, las seis entidades planearon **`$9.343,83 MM`** ajustados en vigencia corriente y contrataron **`$7.780,10 MM` (`83,26%`)**, dejando una brecha nominal de **`-$1.563,73 MM`**. La desviación es altamente heterogénea: la subejecución relativa más pronunciada ocurre en **ANCP (`52,64%`)**, **MinHacienda (`58,26%`)** y **MinAmbiente (`65,00%`)**, mientras que **MinEducación (`88,19%`)**, **DNP (`90,84%`)** y **MinCulturas (`100,44%`)** exhiben mayor cierre global. Los picos anuales superiores al `100%` (como **MinHacienda** en 2022 con `157,4%` y 2025 con `165,9%`, o **MinCulturas** en 2023 con `128,9%`) obedecen a que `valor_del_contrato` en SECOP II contabiliza el compromiso plurianual completo en el año de firma, sumado a adiciones hasta del `50%` (Ley 80/1993, art. 40) y créditos adicionales del Presupuesto General de la Nación.

### 4.2 Peso de la contratación directa: número de contratos frente a recursos comprometidos

In [8]:
fig_directa = graficar_contratacion_directa_entidades(df_resumen)
fig_directa

**Interpretación (Indicador 2 — Contratación directa):** Medir únicamente el conteo de contratos llevaría a concluir que toda la muestra opera de forma idéntica, pues entre el **`87,0%` (MinCulturas)** y el **`97,7%` (MinAmbiente)** de las firmas corresponden a contratación directa (contratos individuales de prestación de servicios profesionales y de apoyo a la gestión, Ley 1150/2007, art. 2, num. 4h). En cambio, al ponderar por **valor en pesos**, emergen dos perfiles institucionales claros: en **MinHacienda (`37,2%`)** y **MinCulturas (`46,8%`)** la mayoría del dinero se adjudica por modalidades competitivas o regímenes focalizados (seguros, administración fiduciaria FONPET, obra/patrimonio), mientras que en **ANCP (`91,4%`)** y **MinEducación (`80,6%`)** predominan los convenios interadministrativos y encargos directos.

### 4.3 Concentración del gasto por proveedor adjudicado (`CR1`, `CR5` e Índice `HHI`)

In [9]:
fig_hhi = graficar_concentracion_hhi(tablas_ind["concentracion"])
fig_hhi

In [10]:
top3_por_entidad = (
    tablas_ind["top_proveedores"]
    .query("ranking_entidad <= 3")[
        [
            "sigla_entidad",
            "ranking_entidad",
            "nombre_proveedor",
            "n_contratos",
            "valor_total_cop",
            "participacion_entidad_pct",
            "modalidad_frecuente",
        ]
    ]
    .assign(valor_total_mm=lambda d: d["valor_total_cop"] / 1e9)
    .drop(columns=["valor_total_cop"])
)
display(top3_por_entidad)

,sigla_entidad,ranking_entidad,nombre_proveedor,n_contratos,participacion_entidad_pct,modalidad_frecuente,valor_total_mm
0,ANCP,1,"SIMPLIFAE PORTUGAL, S.A.",5,19.40,Contratación directa,33.81
1,ANCP,2,COUPA SOFTWARE INC,4,11.38,Contratación directa,19.83
2,ANCP,3,COMCEL S.A.,3,6.00,Selección abreviada,10.46
10,MinHacienda,1,LA PREVISORA S.A. COMPAÑÍA DE SEGUROS,5,12.80,Licitación pública y AMP,94.88
11,MinHacienda,2,CONSORCIO PRAG FONPET 2023,1,8.42,Selección abreviada,62.45
12,MinHacienda,3,CONSORCIO PAC FONPET 2025,1,8.41,Licitación pública y AMP,62.37
20,MinEducación,1,FINANCIERA DE DESARROLLO TERRITORIAL S.A.,4,35.51,Contratación directa,"1,248.73"
21,MinEducación,2,INSTITUTO COLOMBIANO DE CREDITO EDUCATIVO Y ES...,5,8.73,Contratación directa,307.10
22,MinEducación,3,AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO V...,1,7.72,Contratación directa,271.32
30,MinAmbiente,1,PROGRAMA DE LAS NACIONES UNIDAS PARA EL DESARR...,6,25.14,Régimen especial,159.70


**Interpretación (Indicador 3 — Concentración de proveedores):** Frente a los umbrales antimonopolio estándar del *Department of Justice / Federal Trade Commission* y la Superintendencia de Industria y Comercio (`HHI < 1.500` baja, `1.500–2.500` moderada, `> 2.500` alta), **22 de las 24 observaciones entidad-año operan en baja concentración**. Los únicos dos episodios de alta concentración ocurren en 2023 y no corresponden a acaparamiento privado en licitaciones, sino a **ejecución delegada mediante articuladores públicos o multilaterales**: **MinEducación 2023 (`HHI = 5.714,7`, `CR1 = 75,3%`)** por contratos interadministrativos de infraestructura educativa con **FINDETER** (`$1.248,73 MM`) y **MinAmbiente 2023 (`HHI = 2.716,2`, `CR1 = 51,6%`)** por convenios ambientales con el **PNUD** (`$159,70 MM`).

### 4.4 Brechas sectoriales según el Clasificador de Bienes y Servicios UNSPSC

In [11]:
fig_unspsc = graficar_top_brechas_unspsc(tablas_ind["segmentos_unspsc"], top_n=10)
fig_unspsc

**Interpretación (Indicador 4 — Segmentos UNSPSC):** Entre los diez segmentos de mayor volumen, la subejecución acumulada se concentra en **`80 - Servicios de Gestión y Profesionales` (`-$1.334,0 MM`)**, **`84 - Servicios Financieros y de Seguros` (`-$926,0 MM`)** y **`93 - Servicios Políticos y de Asuntos Cívicos` (`-$433,0 MM`)**, mientras que coexiste una sobreejecución en **`72 - Servicios de Edificación y Construcción` (`+$547,8 MM`)**, **`86 - Servicios Educativos` (`+$277,1 MM`)**, **`82 - Servicios Editoriales y de Diseño` (`+$196,4 MM`)** y **`81 - Servicios de Ingeniería e Investigación` (`+$193,4 MM`)**. Esta coexistencia sugiere que las entidades reasignan recursos entre códigos UNSPSC durante la vigencia, hipótesis que ponemos a prueba formalmente en la Sección 5.

## 5. Uso de IA para Ir Más Allá de la Metodología
Caracterizados los indicadores nominales base, documentamos la consulta estructurada a una IA para superar sus sesgos, implementamos dos mejoras en código mostrando su efecto real y justificamos qué descartamos.

---

### 5.1 Prompt formulado y enviado a la IA (contexto, metodología actual, objetivo y restricciones)

```text
Actúa como un economista experto en contratación pública y econometría aplicada en Colombia.

CONTEXTO DEL PROYECTO:
Estamos desarrollando un estudio reproducible en Python sobre las desviaciones entre la planeación contractual (Plan Anual de Adquisiciones - PAA, dataset 9sue-ezhx) y la ejecución contractual efectiva (Contratos Electrónicos SECOP II, dataset jbjy-vk9h) entre 2022 y 2025 para 6 entidades nacionales del nivel central (ANCP, DNP, MinAmbiente, MinCulturas, MinEducación, MinHacienda). La base limpia integra 27.160 ítems del PAA ($9.343,83 mil millones COP planeados ajustados) y 25.317 contratos ($7.780,10 mil millones COP ejecutados), cruzados al 99,8%+ con el Clasificador UNSPSC v14.1 a nivel de Clase (6 dígitos) y Segmento (2 dígitos).

METODOLOGÍA ACTUAL YA IMPLEMENTADA (RECOMENDACIONES BASE DE LOS PROFESORES):
1. Limpieza auditada: rescate de modalidades 'No Definido' desde causal_de_contratacion (Ley 1150/2007), marcación de contratos sin cuantía ($0 en Acuerdos Marco y cuantías simbólicas <= $1.000 COP) y detección de cuasi-duplicados administrativos >= $50.000 millones COP en el PAA.
2. Indicadores descriptivos nominales por entidad y año: (a) tasa de ejecución (% ejecutado / planeado vigencia actual), (b) peso de la contratación directa en número y valor, (c) concentración por proveedor (CR1, CR5 e Índice HHI en escala 0–10.000), y (d) brechas en pesos corrientes por segmento UNSPSC.

OBJETIVO Y RESTRICCIONES:
Propón entre 3 y 4 mejoras metodológicas rigurosas desde la teoría económica o el análisis estadístico de compras públicas que vayan MÁS ALLÁ de los indicadores descriptivos nominales anteriores para responder mejor en dónde y por qué se desvía la ejecución frente a la planeación. Ten en cuenta estas restricciones reales:
- El panel agregado tiene N = 24 observaciones entidad-año (6 entidades × 4 años), aunque a nivel micro tenemos decenas de miles de contratos con fecha exacta de firma, modalidad, proveedor y segmento UNSPSC.
- No existe un identificador único oficial que conecte 1 a 1 una línea del PAA con un id_contrato individual.
- Las mejoras deben poder implementarse en Python (pandas/numpy) sin requerir fuentes privadas.
Para cada mejora propuesta indica: (1) qué sesgo o limitación corrige, (2) su formulación matemática exacta y (3) sus posibles riesgos o supuestos.
```

---

### 5.2 Respuesta completa recibida de la IA (mostrada de forma íntegra y literal)

Desde la economía de las compras públicas y la econometría aplicada al sector público colombiano, los cuatro indicadores descriptivos actuales caracterizan bien *cuánto* se ejecuta y *quién* recibe los recursos en términos nominales, pero enfrentan cuatro sesgos analíticos para explicar **en dónde y por qué** ocurren las desviaciones: (i) distorsión inflacionaria intertemporal, (ii) compensación agregada entre rubros, (iii) pérdida de grados de libertad en el panel macro ($N = 24$) frente a dinámicas estacionales del ciclo presupuestal y (iv) imposibilidad de separar el margen extensivo del intensivo ante la ausencia de llave única PAA–Contrato.

A continuación propongo **cuatro mejoras metodológicas** rigurosas, ordenadas de menor a mayor exigencia de supuestos e implementables íntegramente en Python (`pandas`/`numpy`) con información pública oficial.

### 1. Deflactación Intertemporal a Precios Constantes (IPC DANE, base $2025 = 100$)

1. **Sesgo o limitación que corrige:**
   Corrige el **sesgo de ilusión monetaria y distorsión en la agregación plurianual**. Entre 2022 y 2025 Colombia experimentó un choque inflacionario agudo (inflación anual certificada por el DANE de 13,12% en 2022 y 9,28% en 2023, moderándose en 2024–2025). Sumar directamente los \$9.343,83 mil millones COP planeados y \$7.780,10 mil millones COP ejecutados en pesos corrientes subpondera en poder adquisitivo real las brechas de 2022–2023 frente a las de 2024–2025 y sesga al alza el crecimiento aparente del gasto y de las brechas nominales por segmento UNSPSC.

2. **Formulación matemática exacta:**
   Sea $I_t$ el Índice de Precios al Consumidor oficial del DANE (cierre de diciembre o promedio anual) para la vigencia $t \in \{2022, 2023, 2024, 2025\}$, fijando como año base $t_0 = 2025$. El factor de actualización a pesos constantes de 2025 es $d_t = I_{2025} / I_t$. Para cada entidad $i$, segmento UNSPSC $s$ y año $t$, el valor planeado real ($P_{ist}^{\text{real}}$), el ejecutado real ($E_{ist}^{\text{real}}$) y la brecha real acumulada en el cuatrienio ($\Delta_{is}^{\text{real}}$) se definen como:

   $$P_{ist}^{\text{real}} = P_{ist}^{\text{nom}} \cdot \frac{I_{2025}}{I_t}, \qquad E_{ist}^{\text{real}} = E_{ist}^{\text{nom}} \cdot \frac{I_{2025}}{I_t}$$

   $$\Delta_{is}^{\text{real}} = \sum_{t=2022}^{2025} \left(E_{ist}^{\text{real}} - P_{ist}^{\text{real}}\right) = \sum_{t=2022}^{2025} \frac{I_{2025}}{I_t}\left(E_{ist}^{\text{nom}} - P_{ist}^{\text{nom}}\right)$$

   En `pandas` basta con mapear un diccionario de cuatro escalares anuales del DANE mediante `Series.map()` y multiplicar vectorialmente las series monetarias.

3. **Riesgos o supuestos:**
   - *Invarianza algebraica del cociente intra-anual:* Al multiplicar numerador y denominador del mismo año por el escalar $d_t$, la tasa de ejecución anual $E_{it}^{\text{real}} / P_{it}^{\text{real}} = E_{it}^{\text{nom}} / P_{it}^{\text{nom}}$ permanece idéntica. La corrección modifica **brechas absolutas en COP, agregados 2022–2025, participaciones interanuales y tasas de crecimiento real**, salvo que se deflacte cada contrato con el IPC mensual exacto de su fecha de firma frente a enero del PAA.
   - *Representatividad de la canasta:* El IPC mide precios al consumidor urbano, mientras que la compra estatal combina servicios profesionales indexados al salario mínimo (SMLMV) e insumos especializados más cercanos al IPP.

### 2. Índice de Disimilitud Estructural de Duncan ($D_{it}$) sobre la Composición UNSPSC

1. **Sesgo o limitación que corrige:**
   Corrige el **sesgo de compensación o enmascaramiento agregado** de la tasa global de ejecución ($E_{it}/P_{it}$). Una entidad puede registrar 98% de ejecución total —aparentando alta fidelidad al PAA— porque subejecutó proyectos misionales o tecnológicos en unos segmentos UNSPSC y compensó el monto sobreejecutando servicios de gestión o apoyo administrativo en otros (amparada en el carácter indicativo y modificable del PAA bajo el Decreto 1082 de 2015 o por reclasificación de códigos al adjudicar).

2. **Formulación matemática exacta:**
   Para la entidad $i$ en el año $t$, sean $w_{ist}^{P}$ y $w_{ist}^{E}$ las participaciones relativas de cada categoría UNSPSC $s \in \mathcal{S}$ (a nivel de Segmento de 2 dígitos o Clase de 6 dígitos) sobre el presupuesto total planeado y ejecutado de esa vigencia:

   $$w_{ist}^{P} = \frac{P_{ist}}{\sum_{k \in \mathcal{S}} P_{ikt}}, \qquad w_{ist}^{E} = \frac{E_{ist}}{\sum_{k \in \mathcal{S}} E_{ikt}}$$

   El Índice de Disimilitud (Duncan & Duncan, 1955) adaptado a reasignación presupuestal se expresa como:

   $$D_{it} = \frac{1}{2} \sum_{s \in \mathcal{S}} \left| w_{ist}^{E} - w_{ist}^{P} \right|, \qquad D_{it} \in [0, 1]$$

   Cada sumando $c_{ist} = \frac{1}{2} |w_{ist}^{E} - w_{ist}^{P}|$ mide la contribución aditiva exacta del segmento $s$ al desvío estructural ($\sum_{s} c_{ist} = D_{it}$). Económicamente, $D_{it}$ indica **qué fracción del gasto ejecutado (de 0% a 100%) tendría que reasignarse entre segmentos UNSPSC para replicar exactamente la canasta planeada en el PAA**. Se implementa en `pandas`/`numpy` con agregaciones `groupby`, `merge(..., how="outer").fillna(0)` y `np.abs()`.

3. **Riesgos o supuestos:**
   - *Invarianza de escala:* $D_{it}$ evalúa fidelidad en la *composición relativa*, no en el *nivel de gasto*. Si una entidad recorta o infla todos sus segmentos exactamente en la misma proporción, $D_{it} = 0$; por ello debe leerse en conjunto con la tasa de ejecución agregada (por ejemplo, en un plano cartesiano Tasa de Ejecución vs. $D_{it}$).
   - *Sensibilidad a la granularidad UNSPSC:* Evaluado en Clase (6 dígitos), $D_{it}$ puede sobredimensionar la desviación cuando el comprador corrige códigos contiguos dentro de una misma familia funcional; evaluado en Segmento (2 dígitos, 56 grupos), aísla verdaderos giros sectoriales pero atenúa sustituciones finas intra-segmento.

### 3. Concentración Estacional Intra-anual («Fiebre de Fin de Año») y Ciclo Institucional

1. **Sesgo o limitación que corrige:**
   Supera la **limitación estadística de $N = 24$ celdas entidad-año** aprovechando los 25.317 contratos a nivel micro y su fecha exacta de firma para explicar el *porqué institucional* de las desviaciones. En la teoría de contratación pública (Liebman & Mahoney, *American Economic Review*, 2017), el principio presupuestal de anualidad (Decreto 111 de 1996) genera incentivos de expiración de saldos (*use-it-or-lose-it*) que aceleran contrataciones de menor calidad competitiva al cierre fiscal. En Colombia se suma además el choque exógeno de la **Ley de Garantías Electorales (Ley 996 de 2005)** en el primer semestre de 2022, que restringió convenios interadministrativos y contratación directa antes de elecciones nacionales, forzando picos anticipados en enero de 2022.

2. **Formulación matemática exacta:**
   Sea $E_{itm}$ el monto (o conteo) contratado por la entidad $i$ en el año $t$ y mes de firma $m \in \{1, \dots, 12\}$, y $q_{itm} = E_{itm} / \sum_{r=1}^{12} E_{itr}$ la cuota mensual dentro del año. Se proponen dos métricas complementarias calculables también por subgrupos de modalidad (Directa vs. Competitiva):

   1. *Índice de Divergencia Estacional de Kullback-Leibler normalizado (frente a flujo uniforme $1/12$):*

      $$S_{it} = \frac{1}{\ln(12)} \sum_{m=1}^{12} q_{itm} \ln\left(12 \, q_{itm}\right), \qquad S_{it} \in [0, 1]$$

      adoptando $0 \ln 0 = 0$ (donde $S_{it} = 0$ denota ejecución perfectamente uniforme en el año y $S_{it} = 1$ concentración total en un único mes).
   2. *Razón de Intensidad en Ventanas Críticas ($\text{RIC}$):* Para medir agotamiento de cierre en el cuarto trimestre ($m \in \{10,11,12\}$) —o análogamente anticipación pre-veda en enero ($m = 1$)— frente al promedio mensual del resto de la vigencia:

      $$\text{RIC}_{it}^{\text{Q4}} = \frac{\frac{1}{3}\sum_{m=10}^{12} E_{itm}}{\frac{1}{9}\sum_{m=1}^{9} E_{itm}}$$

   En `pandas`/`numpy` se obtiene extrayendo el mes con `.dt.month` y aplicando `np.where(q > 0, q * np.log(12 * q), 0)`.

3. **Riesgos o supuestos:**
   - *Estacionalidad operativa legítima:* En ministerios del nivel central, firmar contratos anuales de prestación de servicios y pólizas en enero–febrero obedece a continuidad operativa y no a improvisación; por ello, el efecto de Ley de Garantías debe identificarse por contraste interanual (enero 2022 vs. eneros 2023–2025).
   - *Sensibilidad a megacontratos plurianuales:* Un solo contrato con vigencias futuras (Ley 819 de 2003) firmado en noviembre o diciembre puede inflar $\text{RIC}_{it}^{\text{Q4}}$ en dinero, por lo que debe reportarse simultáneamente en **valor** y en **número de contratos**.

### 4. Emparejamiento Micro Bloqueado por Distancia Compuesta (Asignación Bipartita PAA–Contrato)

1. **Sesgo o limitación que corrige:**
   Enfrenta la **ausencia de identificador único 1 a 1 entre líneas del PAA y contratos de SECOP II**. Sin cruce micro, la brecha agregada confunde el *margen extensivo* (proyectos planeados que se cancelan y contratos imprevistos fuera del PAA) con el *margen intensivo* (sobrecostos, adiciones o ahorros de licitación sobre ítems efectivamente ejecutados).

2. **Formulación matemática exacta:**
   Inspirado en el enlace probabilístico con bloqueo exacto (Fellegi & Sunter, 1969), se particionan ambas bases en celdas estrictas $b = (i, t, c_6, \tilde{m})$ definidas por entidad $i$, año $t$, Clase UNSPSC a 6 dígitos $c_6$ y macro-modalidad homologada $\tilde{m}$. Para cada par candidato $(j, k) \in \mathcal{P}_b \times \mathcal{C}_b$ con cuantías positivas ($P_j > 0, E_k > 0$), se define la discrepancia proporcional simétrica en logaritmos:

   $$d(j, k) = \left| \ln(E_k) - \ln(P_j) \right|$$

   Dentro de cada celda $b$, se busca la asignación uno a uno sin reemplazo $x_{jk} \in \{0, 1\}$ acotada por un umbral económico máximo $\tau > 0$ (por ejemplo, $\tau = \ln(1{,}5) \approx 0{,}405$, consistente con el tope legal del 50% para adiciones contractuales del art. 40 de la Ley 80 de 1993):

   $$\min_{\mathbf{X} \in \{0,1\}^{|\mathcal{P}_b| \times |\mathcal{C}_b|}} \sum_{j \in \mathcal{P}_b} \sum_{k \in \mathcal{C}_b} d(j, k) \, x_{jk} \quad \text{s.a.} \quad \sum_{k \in \mathcal{C}_b} x_{jk} \le 1, \quad \sum_{j \in \mathcal{P}_b} x_{jk} \le 1, \quad x_{jk} = 0 \text{ si } d(j,k) > \tau$$

   En `pandas`/`numpy` puede aproximarse determinísticamente ordenando pares candidatos por $d(j,k)$ ascendente dentro de cada bloque $b$ (emparejamiento *greedy* sin reemplazo), separando tres componentes: ítems PAA huérfanos ($x_{j\cdot}=0$), contratos emergentes ($x_{\cdot k}=0$) y brecha intensiva en pares emparejados ($\sum x_{jk}(E_k - P_j)$).

3. **Riesgos o supuestos:**
   - *Violación frecuente de la cardinalidad 1 a 1 (relaciones $1:M$ y $M:1$):* En la práctica presupuestal colombiana, una única línea agregada del PAA (p. ej., una bolsa global para contratistas de apoyo por \$2.000 millones COP) suele ejecutarse mediante decenas de contratos individuales de \$50 millones COP ($1:M$), o múltiples líneas del PAA se consolidan en un solo proceso por lotes ($M:1$). En ambos casos reales $|\ln(E_k) - \ln(P_j)| > \tau$, produciendo **falsos negativos sistemáticos** que sobrestiman artificialmente tanto la "no ejecución" como la "improvisación contractual".
   - *Indistinguibilidad y reclasificación:* En celdas densas con múltiples contratos de prestación de servicios de idéntico valor y Clase UNSPSC, el emparejamiento individual es arbitrario; a la inversa, si al adjudicar cambia la modalidad o el código de Clase frente al PAA, el bloqueo exacto descarta pares verdaderos. Por tanto, solo constituye un ejercicio exploratorio de sensibilidad y no equivale a trazabilidad contable oficial.

### Síntesis Comparativa de Viabilidad Analítica

| Mejora propuesta | Fenómeno económico que aísla | Nivel de datos | Robustez de supuestos |
|---|---|---|---|
| **1. Deflactación IPC DANE** | Poder adquisitivo real y brechas comparables 2022–2025 | Panel / Segmento | **Alta** (supuesto estándar de precios constantes) |
| **2. Índice de Duncan ($D_{it}$)** | Reasignación composicional oculta entre rubros UNSPSC | Entidad × Año × UNSPSC | **Alta** (descomposición contable exacta en $[0, 1]$) |
| **3. Estacionalidad micro ($S_{it}$, $\text{RIC}$)** | Premura de cierre anual (anualidad) y choque Ley de Garantías | Micro (`25.317` contratos) | **Media-Alta** (requiere separar eneros operativos y vigencias futuras) |
| **4. Emparejamiento bloqueado** | Separación tentativa de margen extensivo vs. intensivo | Micro bipartito PAA–SECOP | **Baja-Media** (vulnerable a bolsas $1:M$ y consolidación $M:1$) |

---

### 5.3 Identificación precisa de qué sugerencias van más allá de lo recomendado por los profesores

Las recomendaciones base del equipo pedagógico (`R1–R8`) establecían indicadores descriptivos expresados en pesos corrientes (`% ejecutado vs. planeado`, peso de la contratación directa, concentración por proveedor y brechas UNSPSC). Las cuatro propuestas de la IA superan ese marco base en cuatro dimensiones analíticas nuevas:

| Propuesta de la IA | Qué pedía la metodología recomendada base | En qué va más allá la sugerencia de la IA |
|---|---|---|
| **1. Deflactación IPC DANE (`2025 = 100`)** | Sumar y restar montos `2022–2025` en pesos corrientes nominales. | Corrige la ilusión monetaria tras el choque inflacionario colombiano (`13,12%` en 2022 y `9,28%` en 2023), expresando brechas y tasas de crecimiento en poder adquisitivo constante de 2025. |
| **2. Índice de Disimilitud de Duncan ($D_{it}$)** | Medir la tasa global `Ejecutado / Planeado` y ver brechas aisladas por rubro. | Introduce una métrica acotada en $[0, 1]$ que cuantifica la **fracción exacta del presupuesto reasignada entre segmentos UNSPSC**, destapando el sesgo de compensación cuando subejecuciones y sobreejecuciones se anulan en el agregado. |
| **3. Divergencia estacional Kullback-Leibler ($S_{it}$)** | Agregación anual (`N = 24` celdas entidad-año). | Aprovecha la fecha exacta de firma a nivel micro para medir entropía mensual, agotamiento de saldos en Q4 (*use-it-or-lose-it*) y anticipación por Ley de Garantías 2022. |
| **4. Emparejamiento bipartito `1:1` PAA–Contrato** | Cruce agregado por `Entidad × Año × Segmento UNSPSC`. | Plantea un algoritmo tipo Fellegi-Sunter con bloqueo exacto y cota logarítmica para intentar separar margen extensivo e intensivo sin llave primaria. |

In [12]:
# 5.4 Implementación en código de las Mejoras #1 (IPC DANE 2025=100) y #2 (Índice de Duncan)
factores_ipc_df = pd.DataFrame(
    [
        {"anio": anio, "factor_actualizacion_a_pesos_2025": factor}
        for anio, factor in FACTOR_DEFLACTOR_IPC_2025.items()
    ]
)
display(factores_ipc_df)

df_antes_despues = tablas_ind["rubro5_antes_despues"]
display(
    df_antes_despues[
        [
            "sigla_entidad",
            "brecha_nominal_mm",
            "brecha_real_2025_mm",
            "cambio_brecha_real_vs_nominal_mm",
            "crecimiento_ejec_nominal_2022_2025_pct",
            "crecimiento_ejec_real_2022_2025_pct",
            "sesgo_inflacionario_crecimiento_pp",
            "tasa_ejecucion_nominal_pct",
            "duncan_promedio_pct",
            "duncan_max_pct",
        ]
    ]
)

,anio,factor_actualizacion_a_pesos_2025
0,2022,1.21
1,2023,1.11
2,2024,1.05
3,2025,1.00


,sigla_entidad,brecha_nominal_mm,brecha_real_2025_mm,cambio_brecha_real_vs_nominal_mm,crecimiento_ejec_nominal_2022_2025_pct,crecimiento_ejec_real_2022_2025_pct,sesgo_inflacionario_crecimiento_pp,tasa_ejecucion_nominal_pct,duncan_promedio_pct,duncan_max_pct
0,ANCP,-156.76,-166.34,-9.58,63.04,34.93,28.10,52.64,10.50,22.02
1,MinHacienda,-531.10,-570.03,-38.93,60.98,33.24,27.75,58.26,40.31,55.88
2,MinEducación,-470.85,-476.97,-6.13,282.73,216.76,65.97,88.19,24.82,28.38
3,MinAmbiente,-341.97,-377.66,-35.68,-2.80,-19.56,16.75,65.00,20.18,52.05
4,DNP,-71.75,-80.51,-8.76,-2.41,-19.23,16.82,90.84,13.70,25.43
5,MinCulturas,8.70,14.03,5.34,165.66,119.87,45.79,100.44,32.07,45.14
6,TOTAL_MUESTRA,"-1,563.73","-1,657.48",-93.75,131.66,91.73,39.93,83.26,23.60,55.88


In [13]:
fig_r5_ipc = graficar_rubro5_antes_despues_ipc(df_antes_despues)
fig_r5_ipc

In [14]:
fig_r5_duncan = graficar_rubro5_ejecucion_vs_duncan(df_resumen)
fig_r5_duncan

### 5.5 Efecto empírico sobre los resultados y evaluación crítica de descartes

#### A. Efecto demostrable *Antes vs. Después* de las dos mejoras implementadas
1. **Efecto de la Deflactación IPC DANE (Mejora #1):**
   - **Brecha real más profunda:** al actualizar los montos de `2022–2024` a pesos constantes de 2025, la subejecución acumulada de las seis entidades pasa de **`-$1.563,73 MM` nominales (Antes)** a **`-$1.657,48 MM` reales (Después)**: la contabilidad nominal ocultaba **`$93,75 mil millones COP`** de brecha en poder adquisitivo real (concentrados sobre todo en **MinHacienda**, con `-$38,93 MM` adicionales, y **MinAmbiente**, con `-$35,68 MM`).
   - **Corrección de ilusión monetaria en la tendencia:** el gasto ejecutado parecía crecer **`+131,66%`** entre 2022 y 2025 en pesos corrientes, pero en términos reales creció **`+91,73%` (`-39,93 pp` de sesgo inflacionario)**. Más importante aún, en **DNP** y **MinAmbiente** una variación nominal casi plana (`-2,41%` y `-2,80%`) se revela como una **contracción real severa de `-19,23%` y `-19,56%`**.
2. **Efecto del Índice de Disimilitud de Duncan (Mejora #2):**
   - En promedio, las seis entidades reasignaron el **`23,60%`** de su presupuesto ejecutado entre segmentos UNSPSC frente a la canasta planeada en el PAA.
   - El caso más revelador es **MinCulturas**: bajo el indicador nominal base parecía cumplir casi exactamente su plan (**`100,44%`** de ejecución acumulada), pero su **Índice de Duncan promedia `32,07%` (con pico de `45,14%` en 2023)**, demostrando que casi un tercio de sus recursos cambió de sector UNSPSC por compensación interna entre rubros. En **MinHacienda**, la reasignación estructural alcanza **`40,31%` promedio (`55,88%` en 2023)**.

#### B. Evaluación crítica: qué sugerencias de la IA descartó el equipo y por qué
1. **Descartamos la Mejora #4 (Emparejamiento micro bipartito `1:1` PAA–Contrato con $|\ln E_k - \ln P_j| \le \ln(1,5)$):**
   - **Justificación técnica e institucional:** en la práctica presupuestal de los ministerios colombianos, una sola línea del PAA opera frecuentemente como una **bolsa global pluripersonal (`1:M`)** —por ejemplo, un renglón planeado por `$2.000 millones` que financia 40 contratos individuales de prestación de servicios de `$50 millones` cada uno—, o varias líneas de insumos se consolidan en un único proceso licitatorio por lotes (`M:1`). Imponer una asignación matemática `1:1` con tolerancia máxima del `50%` ($|\ln(50) - \ln(2000)| = 3,69 \gg 0,405$) rechazaría sistemáticamente ejecuciones `1:M` completamente legítimas, fabricando miles de falsos "ítems abandonados" y falsos "contratos improvisados" que distorsionarían la veeduría.
2. **Descartamos la Mejora #3 (Divergencia estacional intra-anual de Kullback-Leibler $S_{it}$ y $\text{RIC}^{\text{Q4}}$):**
   - **Justificación metodológica y de foco (Rubro 4):** primero, el PAA (`9sue-ezhx`) no contiene un cronograma mensual vinculante de desembolsos (solo texto indicativo en `fecha_esperada_de_inicio`), por lo que la estacionalidad mensual de firmas mide el ritmo operativo y no una desviación frente al PAA. Segundo, en entidades del nivel central el pico masivo de firmas en enero obedece a la contratación anual ordinaria de apoyo profesional —reforzado en enero de 2022 por el cierre previo a la Ley de Garantías Electorales (Ley 996 de 2005)—; incluirlo abriría una pregunta distinta sobre calendario electoral y añadiría un componente superfluo frente a nuestra pregunta central.

## 6. Síntesis de Hallazgos, Limitaciones Explícitas y Tablero en Streamlit
Integrando los indicadores base y las mejoras con precios constantes e Índice de Duncan, respondemos de forma directa la pregunta de investigación, explicitamos las limitaciones del estudio y conectamos el tablero.

### Q&A
- **¿Qué tanto se desvía la contratación ejecutada en SECOP II frente a lo planeado en el PAA (2022–2025)?** En las seis entidades nacionales analizadas, la ejecución acumulada alcanza el **`83,26%`** en pesos corrientes (`$7.780,10 MM` ejecutados frente a `$9.343,83 MM` planeados ajustados) y el **`83,47%`** en pesos constantes de 2025 (`$8.371,59 MM` frente a `$10.029,07 MM`), con una subejecución real de **`-$1.657,48 MM`** y una reasignación composicional promedio del **`23,60%`** entre segmentos UNSPSC según el Índice de Duncan.
- **¿En qué entidades, modalidades, proveedores o segmentos UNSPSC se concentra dicha desviación?** En **entidades**, las mayores brechas relativas de subejecución se concentran en **ANCP (`52,6%`)**, **MinHacienda (`58,3%`, con `40,3%` de reasignación Duncan)** y **MinAmbiente (`65,0%`)**, mientras que **MinCulturas** exhibe compensación oculta (`100,4%` global pero `32,1%` de giro sectorial). En **modalidades**, la contratación directa agrupa el `91,5%` de los contratos pero el `65,1%` del valor total (desde `37,2%` en MinHacienda hasta `91,4%` en ANCP). En **proveedores**, la desconcentración predomina en `22/24` celdas (`HHI < 1.500`), concentrándose solo en 2023 mediante articuladores institucionales (**FINDETER** en MinEducación con `HHI = 5.714,7` y **PNUD** en MinAmbiente con `HHI = 2.716,2`). En **segmentos UNSPSC**, la subejecución se concentra en servicios de gestión (`80`), financieros (`84`) y cívicos (`93`), compensada parcialmente por sobreejecución en edificación (`72`), educación (`86`), diseño editorial (`82`) e ingeniería (`81`).

### Data Analysis Key Findings
- **Calidad y homologación crítica antes de medir:** rescatar modalidades desde la causal legal recuperó `2.251` ítems del PAA (`$1.975,68 MM`), controlar la reedición atípica de `$333,54 MM` en MinAmbiente 2023 corrigió `+18,0 pp` su tasa de ejecución, separar `964` contratos en `$0` o cuantía simbólica evitó sesgar medias e índices `HHI`, y cruzar UNSPSC a **Clase (6 dígitos) / Segmento (2 dígitos)** elevó el emparejamiento de `< 40%` a **`99,99%` (PAA) y `99,80%` (Contratos)**.
- **Concentración extrema por tamaño contractual:** apenas `708` contratos de gran cuantía (`> $1.000M`, el `2,8%` del total) explican el **`75,4%` de los recursos ejecutados** (`$5.866,32 MM`).
- **Valor analítico de las extensiones con IA:** deflactar con IPC DANE descubrió **`-$93,75 MM`** adicionales de brecha real y corrigió `-39,93 pp` de ilusión inflacionaria en el crecimiento `2022–2025`, mientras que el Índice de Duncan evidenció desvíos sectoriales severos invisibles para el cociente agregado `Ejecutado / Planeado`.

### Insights or Next Steps
- **Limitaciones metodológicas explícitas (alcance, supuestos y sesgos):** (1) **Última foto anual vs. compromiso plurianual:** `datos.gov.co` publica únicamente la última versión modificada del PAA de cada año, mientras que `jbjy-vk9h` registra el valor total del contrato en su fecha de firma (incluyendo vigencias futuras y adiciones legales), por lo que tasas `> 100%` no implican gasto sin apropiación presupuestal; (2) **Ausencia de llave `1:1` PAA–Contrato:** la existencia de bolsas pluripersonales (`1:M`) impide auditar cancelaciones individuales ítem por ítem sin acceso al aplicativo interno SIIF Nación; (3) **Acuerdos Marco en `$0`:** en la ANCP el instrumento rector se firma por `$0` y la compra ocurre en cada entidad compradora; (4) **Alcance descriptivo y validez externa:** los indicadores constituyen alertas tempranas de gestión y veeduría ciudadana —**no prueban causalidad ni irregularidades disciplinarias o fiscales**— y sus conclusiones se circunscriben a ministerios/agencias del nivel central nacional, sin extrapolarse a municipios, departamentos ni entidades desconcentradas.
- **Exploración interactiva en el tablero Streamlit (`R8`):** todos los indicadores, filtros por entidad/año, selector entre pesos constantes de 2025 y corrientes, tablas de proveedores y descargas CSV están integrados en `app_streamlit.py` (ejecutable desde la raíz del proyecto con `streamlit run app_streamlit.py`).